In [1]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import RobertaPreTrainedModel, RobertaModel, RobertaTokenizer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, precision_recall_fscore_support
import joblib
import re
import warnings
warnings.filterwarnings('ignore')

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch device: {device}")
print(f"CUDA available: {torch.cuda.is_available()}")

2025-07-17 15:42:07.409937: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1752766927.579903      36 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1752766927.638098      36 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


PyTorch device: cuda
CUDA available: True


In [7]:
# load data
# file_path = "/kaggle/input/human-memory-and-cognition/hippoCorpusV2.csv"
file_path = "/kaggle/input/hippoCorpusV2.csv"
df = pd.read_csv(file_path)
df = df[['story', 'stressful']].dropna()

# Convert to binary labels
def map_stress_to_binary(stress_level):
    return 0 if stress_level == 1 else 1

df['labels'] = df['stressful'].apply(map_stress_to_binary).astype(int)

print(f"Dataset: {df.shape}, Labels: 0={sum(df['labels']==0)}, 1={sum(df['labels']==1)}")

Dataset: (6854, 3), Labels: 0=3125, 1=3729


In [8]:
class SimpleLIWCExtractor:
    """Simplified LIWC extractor for integration"""
    
    def __init__(self):
        print("Initializing Simple LIWC Extractor...")
        
        # Key psychological categories for stress detection
        self.keyword_dict = {
            'posemo': [
                'happy', 'joy', 'love', 'excellent', 'good', 'great', 'wonderful', 'amazing',
                'fantastic', 'perfect', 'beautiful', 'awesome', 'brilliant', 'delighted',
                'pleased', 'excited', 'cheerful', 'optimistic', 'satisfied', 'grateful',
                'proud', 'confident', 'blessed', 'peaceful', 'content', 'energized'
            ],
            'negemo': [
                'sad', 'angry', 'hate', 'terrible', 'awful', 'horrible', 'bad', 'upset',
                'disgusting', 'annoying', 'disappointing', 'frustrating', 'depressing',
                'miserable', 'furious', 'irritated', 'disturbed', 'worried', 'concerned',
                'negative', 'wrong', 'fail', 'problem', 'difficult', 'hard', 'tough'
            ],
            'anx': [
                'anxious', 'worried', 'nervous', 'stress', 'tension', 'panic', 'fear',
                'afraid', 'scared', 'terrified', 'overwhelmed', 'restless', 'uneasy',
                'apprehensive', 'concerned', 'distressed', 'troubled', 'agitated',
                'pressure', 'strain', 'burden', 'overwhelming', 'stressful', 'crushing'
            ],
            'anger': [
                'angry', 'mad', 'furious', 'rage', 'irritated', 'annoyed', 'frustrated',
                'outraged', 'livid', 'enraged', 'infuriated', 'resentful', 'bitter',
                'hostile', 'aggressive', 'violent', 'hate', 'despise', 'loathe'
            ],
            'sad': [
                'sad', 'depressed', 'unhappy', 'miserable', 'down', 'blue', 'crying',
                'tears', 'weeping', 'grief', 'sorrow', 'melancholy', 'gloomy',
                'heartbroken', 'devastated', 'disappointed', 'dejected', 'despondent',
                'lonely', 'empty', 'hopeless', 'despair', 'exhausted', 'burned'
            ],
            'work': [
                'work', 'job', 'career', 'office', 'business', 'employee', 'boss',
                'company', 'organization', 'professional', 'workplace', 'deadline',
                'project', 'task', 'meeting', 'supervisor', 'colleague', 'performance'
            ],
            'family': [
                'family', 'mother', 'father', 'parent', 'child', 'sister', 'brother',
                'mom', 'dad', 'son', 'daughter', 'home', 'grandmother', 'grandfather',
                'spouse', 'husband', 'wife', 'kids', 'children'
            ],
            'friend': [
                'friend', 'buddy', 'pal', 'companion', 'colleague', 'mate',
                'friendship', 'classmate', 'roommate', 'neighbor', 'partner',
                'friends', 'conversation', 'social'
            ],
            'achieve': [
                'achieve', 'success', 'goal', 'accomplish', 'win', 'complete', 'finish',
                'succeed', 'victory', 'triumph', 'attain', 'reach', 'obtain',
                'fulfill', 'realize', 'master', 'excel', 'outstanding', 'award',
                'proud', 'accomplished', 'productive'
            ],
            'focuspast': [
                'was', 'were', 'had', 'did', 'yesterday', 'ago', 'before', 'previous',
                'earlier', 'formerly', 'once', 'used', 'past', 'remember', 'recalled'
            ],
            'focusfuture': [
                'will', 'going', 'tomorrow', 'next', 'future', 'plan', 'hope',
                'expect', 'anticipate', 'predict', 'upcoming', 'later', 'eventually',
                'planning', 'excited', 'look'
            ],
            'certain': [
                'always', 'never', 'definitely', 'sure', 'certain', 'absolutely',
                'completely', 'totally', 'obviously', 'clearly', 'undoubtedly',
                'confident', 'exactly', 'precisely'
            ],
            'tentat': [
                'maybe', 'perhaps', 'might', 'could', 'possibly', 'probably',
                'seems', 'appears', 'likely', 'potentially', 'suppose', 'guess',
                'uncertain', 'unclear', 'doubt', 'question'
            ]
        }
        
        self.categories = list(self.keyword_dict.keys())
        self.scaler = StandardScaler()
        self.fitted = False
    
    def extract_features(self, text):
        """Extract LIWC features from text"""
        if pd.isna(text):
            text = ""
        
        text_lower = text.lower()
        words = re.findall(r'\b\w+\b', text_lower)
        total_words = len(words) if words else 1
        
        # Extract LIWC features
        features = []
        for category in self.categories:
            keywords = self.keyword_dict[category]
            count = sum(1 for word in words if word in keywords)
            percentage = (count / total_words) * 100
            features.append(percentage)
        
        # Add basic text statistics
        features.extend([
            len(words),                                    # word_count
            len(text),                                     # char_count
            text.count('!'),                              # exclamation_count
            text.count('?'),                              # question_count
            sum(1 for c in text if c.isupper()) / len(text) if text else 0,  # caps_ratio
        ])
        
        return np.array(features)
    
    def extract_batch_features(self, texts):
        """Extract features for a batch of texts"""
        features = []
        for text in texts:
            features.append(self.extract_features(text))
        return np.array(features)
    
    def fit_transform(self, texts):
        """Fit scaler and transform features"""
        features = self.extract_batch_features(texts)
        scaled_features = self.scaler.fit_transform(features)
        self.fitted = True
        print(f"✓ LIWC features: {features.shape[1]} per text")
        return scaled_features
    
    def transform(self, texts):
        """Transform features using fitted scaler"""
        if not self.fitted:
            raise ValueError("Must fit extractor first")
        features = self.extract_batch_features(texts)
        return self.scaler.transform(features)
    
    def get_feature_count(self):
        """Get number of features extracted"""
        return len(self.categories) + 5  # LIWC categories + 5 basic stats

In [9]:
# Analyze_single_text function
def analyze_single_text(text, label, stress_level):
    """Detailed analysis of a single text"""
    print(f"\n TEXT ANALYSIS:")
    print(f"   Original stress level: {stress_level}")
    print(f"   Binary label: {'STRESSED' if label == 1 else 'NOT STRESSED'}")
    print(f"   Text: \"{text[:150]}...\"")

    # Extract LIWC features (returns numpy array)
    liwc_features_array = extractor.extract_features(text)
    
    # Convert array to dictionary using category names
    liwc_features = {}
    for i, category in enumerate(extractor.categories):
        liwc_features[category] = liwc_features_array[i]

    # Show active features
    active_features = {k: v for k, v in liwc_features.items() if v > 0}

    if active_features:
        print(f"   Psychological patterns detected:")
        for category, percentage in sorted(active_features.items(), key=lambda x: x[1], reverse=True)[:8]:
            print(f"     {category}: {percentage:.1f}%")
    else:
        print(f"   No strong psychological patterns detected")

    # Show which keywords triggered
    print(f"   Keyword matches:")
    text_words = set(text.lower().split())

    for category in list(active_features.keys())[:5]:  # Top 5 categories
        if category in extractor.keyword_dict:
            keywords = extractor.keyword_dict[category]
            found_keywords = []
            for word in text_words:
                if word in keywords:
                    found_keywords.append(word)
            if found_keywords:
                print(f"     {category}: {found_keywords}")

    return liwc_features_array  # Return the original array

In [10]:
# Initialize LIWC feature extractor
extractor = SimpleLIWCExtractor()

Initializing Simple LIWC Extractor...


In [11]:
# Get examples of stressed and non-stressed texts
stressed_examples = df[df['labels'] == 1].sample(3, random_state=42)
non_stressed_examples = df[df['labels'] == 0].sample(3, random_state=42)
print(stressed_examples)

                                                  story  stressful  labels
4954  So I just did something. I created a Subreddit...        3.0       1
6381  it was same time like this, 2 years ago when t...        2.0       1
4982  Dear Diary, There has been so much change in t...        3.0       1


In [12]:
# Get examples of stressed and non-stressed texts
stressed_examples = df[df['labels'] == 1].sample(3, random_state=42)
non_stressed_examples = df[df['labels'] == 0].sample(3, random_state=42)
print(stressed_examples)

# Analyze stressed examples
stressed_features = []
for i, (idx, row) in enumerate(stressed_examples.iterrows()):
    print(f"\n--- STRESSED EXAMPLE {i+1} ---")
    features = analyze_single_text(row['story'], row['labels'], row['stressful'])
    stressed_features.append(features)

# Analyze non-stressed examples
non_stressed_features = []
for i, (idx, row) in enumerate(non_stressed_examples.iterrows()):
    print(f"\n--- NON-STRESSED EXAMPLE {i+1} ---")
    features = analyze_single_text(row['story'], row['labels'], row['stressful'])
    non_stressed_features.append(features)

                                                  story  stressful  labels
4954  So I just did something. I created a Subreddit...        3.0       1
6381  it was same time like this, 2 years ago when t...        2.0       1
4982  Dear Diary, There has been so much change in t...        3.0       1

--- STRESSED EXAMPLE 1 ---

 TEXT ANALYSIS:
   Original stress level: 3.0
   Binary label: STRESSED
   Text: "So I just did something. I created a Subreddit on the GalaxyNote 10. I am really proud of myself because going out on a limb like that isn't something..."
   Psychological patterns detected:
     focusfuture: 2.6%
     focuspast: 1.3%
     tentat: 1.3%
     posemo: 0.6%
     anx: 0.6%
     work: 0.6%
     achieve: 0.6%
     certain: 0.6%
   Keyword matches:
     posemo: ['proud']
     anx: ['nervous']
     work: ['work']
     achieve: ['proud']
     focuspast: ['used', 'did']

--- STRESSED EXAMPLE 2 ---

 TEXT ANALYSIS:
   Original stress level: 2.0
   Binary label: STRESSED
   Text